# NovoCart Data Cube

## Purpose
This notebook creates a **unified data cube** (`novacart_dev.gold.data_cube`) by combining the fact table with all dimension tables. This cube serves as a single source of truth for KPI reporting and analytics.

## What is a Data Cube?
A data cube is a denormalized, pre-joined table that combines:
- **Fact table** (`fact_order_items`): Transaction-level metrics (revenue, quantity, prices)
- **Dimension tables**: Descriptive attributes for analysis
  - `dim_products`: Product details (name, category, origin)
  - `dim_customers`: Customer registration information
  - `dim_dates`: Time-based attributes (year, quarter, month, day of week)
  - `dim_orders`: Order-level details (status, channel, country)

## Benefits
✓ **Simplified queries**: No need to repeatedly join fact and dimension tables  
✓ **Better performance**: Single table scan instead of multiple joins  
✓ **Consistency**: All KPIs use the same underlying data structure  
✓ **Easier maintenance**: Update joins in one place instead of across multiple notebooks  
✓ **Self-service analytics**: Business users can query a single table with all context

## Usage
The data cube is used in the **NovoCart Gold KPIs** notebook to calculate all business metrics.

```python
# Load the data cube
cube = spark.table("novacart_dev.gold.data_cube")

# Example: Calculate revenue by product category
revenue_by_category = cube.filter(F.col("order_status_clean") == "completed") \
    .groupBy("product_category") \
    .agg(F.sum("revenue_usd").alias("total_revenue"))
```

## Key Fields
- **Metrics**: `revenue_usd`, `quantity`, `unit_price`, `line_total`
- **Product**: `product_name`, `product_category`, `product_origin_country`
- **Customer**: `registration_date`, `registration_year`, `registration_month`
- **Time**: `order_year`, `order_quarter`, `order_month`, `day_name`, `is_weekend`
- **Order**: `order_status_clean`, `fact_channel`, `order_country`
- **Data Quality**: All `dq_*` flags for quality monitoring

In [0]:
from pyspark.sql import functions as F

# Load fact and dimension tables
fact = spark.table("novacart_dev.gold.fact_order_items")
dim_products = spark.table("novacart_dev.gold.dim_products")
dim_customers = spark.table("novacart_dev.gold.dim_customers")
dim_dates = spark.table("novacart_dev.gold.dim_dates")
dim_orders = spark.table("novacart_dev.gold.dim_orders")

# Create data cube by joining fact with all dimensions
# Note: date_key join uses fact.date_key = dim_dates.full_date (DATE = DATE)
data_cube = fact \
    .join(dim_products, fact.product_key == dim_products.product_key, "left") \
    .join(dim_customers, fact.customer_key == dim_customers.customer_key, "left") \
    .join(dim_dates, fact.date_key == dim_dates.full_date, "left") \
    .join(dim_orders, fact.order_key == dim_orders.order_key, "left") \
    .select(
        # Fact table keys and metrics
        fact.order_item_key,
        fact.order_key,
        fact.product_key,
        fact.customer_key,
        fact.date_key,
        fact.quantity,
        fact.unit_price,
        fact.line_total,
        fact.revenue_usd,
        fact.order_status_clean,
        fact.channel.alias("fact_channel"),
        fact.order_country,
        fact.category.alias("fact_category"),
        fact.product_currency,
        
        # Data quality flags from fact
        fact.dq_line_total_mismatch,
        fact.dq_orphan_order,
        fact.dq_orphan_product,
        fact.dq_missing_order_item_id,
        fact.dq_invalid_quantity,
        fact.dq_invalid_item_price,
        fact.dq_orphan_customer,
        fact.dq_missing_order_id,
        fact.dq_invalid_order_date,
        fact.dq_invalid_status,
        fact.dq_invalid_amount,
        fact.dq_missing_product_id,
        fact.dq_missing_product_name,
        
        # Product dimension attributes
        dim_products.product_name,
        dim_products.category.alias("product_category"),
        dim_products.price.alias("product_price"),
        dim_products.currency.alias("product_dim_currency"),
        dim_products.product_origin_country,
        
        # Customer dimension attributes
        dim_customers.registration_date,
        dim_customers.registration_year,
        dim_customers.registration_month,
        
        # Date dimension attributes
        dim_dates.full_date,
        dim_dates.year.alias("order_year"),
        dim_dates.quarter.alias("order_quarter"),
        dim_dates.month.alias("order_month"),
        dim_dates.month_name.alias("order_month_name"),
        dim_dates.day.alias("order_day"),
        dim_dates.day_of_week,
        dim_dates.day_name,
        dim_dates.is_weekend,
        
        # Order dimension attributes
        dim_orders.customer_id,
        dim_orders.order_date,
        dim_orders.order_status,
        dim_orders.channel.alias("order_channel"),
        dim_orders.country_code,
        dim_orders.total_amount.alias("order_total_amount"),
        dim_orders.currency.alias("order_currency")
    )

# Save as managed table in gold layer
data_cube.write.mode("overwrite").saveAsTable("novacart_dev.gold.data_cube")

print("✓ Data cube created successfully!")
print(f"  Total records: {data_cube.count():,}")
print(f"  Table: novacart_dev.gold.data_cube")
print(f"\n✓ Data cube combines:")
print(f"  - Fact: fact_order_items")
print(f"  - Dimensions: dim_products, dim_customers, dim_dates, dim_orders")

In [0]:
# Load and display a sample of the data cube
cube_sample = spark.table("novacart_dev.gold.data_cube")

print("Data Cube Schema:")
print(f"Total Columns: {len(cube_sample.columns)}")
print("\nColumn Categories:")
print(f"  - Keys: order_item_key, order_key, product_key, customer_key, date_key")
print(f"  - Metrics: revenue_usd, quantity, unit_price, line_total")
print(f"  - Dimensions: {len([c for c in cube_sample.columns if not c.startswith('dq_') and c not in ['order_item_key', 'order_key', 'product_key', 'customer_key', 'date_key', 'revenue_usd', 'quantity', 'unit_price', 'line_total']])} attribute columns")
print(f"  - Data Quality: {len([c for c in cube_sample.columns if c.startswith('dq_')])} DQ flags")

print("\nSample Records (first 5):")
display(cube_sample.limit(5))